In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys
from openpyxl import Workbook

In [2]:
#Entry
tperiod = 8

# Specify the instance version here

In [3]:
# Specify the instance version
instance_version = "smallsize2" #Versions: "original", "smallsize", "smallsize2"

# Define file names dynamically based on instance version
location_file = f"fullDist Set V Data Location-{instance_version}.xlsx"
county_file = f"fullDist Set C Data-{instance_version}.xlsx"
resource_file = f"fullDist R-{instance_version}.xlsx"
time_matrix_file = f"durationFull-{instance_version}.txt"
distance_matrix_file = f"distanceFull-{instance_version}.txt"
inputs_file = f"fullDistInputs-{instance_version}.xlsx"
k_ijt_file = f"K_ijt_data_fixed-{instance_version}.xlsx"

# Specify the number of scenarios here

In [4]:
# Define the number of scenarios
sce = list(range(1))  # Example: 10 scenarios (adjust as needed)

# Generate equal probabilities for each scenario
P = [1/len(sce)] * len(sce)

#Demand data
demand_file = f"demand_data_{instance_version}.xlsx"

# Dynamically generate sheet names based on the number of scenarios
sheet_names = [f"Demand_{i+1}" for i in range(len(sce))]

print(f"Number of scenarios: {len(sce)}")
print(f"Generated sheet names: {sheet_names}")

Number of scenarios: 1
Generated sheet names: ['Demand_1']


In [5]:
# Location Data
df_V = pd.read_excel(location_file)
print(df_V.head())

# County Data
df_C = pd.read_excel(county_file)
print(df_C.head())

# Resource Data
df_R = pd.read_excel(resource_file)
print(df_R.head())

# Time and Distance Matrices
timeMatrix = np.loadtxt(time_matrix_file)  # Time duration matrix (seconds)
distanceMatrix = np.loadtxt(distance_matrix_file)  # Distance matrix
print(timeMatrix)
print(distanceMatrix)

# Inputs Data
df_inputs = pd.read_excel(inputs_file)
print(df_inputs.head())

# K_ijt Data
df_K = pd.read_excel(k_ijt_file)
print(df_K.head())

   Index      County       Type  Cluster  \
0      0         NaN  Warehouse      NaN   
1      1         NaN   Supplier      NaN   
2      2  Georgetown    Shelter      9.0   
3      3         NaN    Shelter      NaN   
4      4       Horry    Shelter     18.0   

                                            Facility  \
0                     US Military Base - Cot Storage   
1                    FEMA Warehouse (Food and Water)   
2  Cluster 9 (Andrews Elementary School, Andrews ...   
3  Cluster 1 (Pleasant Hill Elementary, Carvers B...   
4                                  Loris High School   

                                             Address   latitude  longitude  \
0                Garners Ferry Rd, Hopkins, SC 29061 -80.833969  34.045263   
1  3780 Southside Industrial Pkwy, Atlanta, GA 30354 -84.373590  33.650208   
2          13072 County Line Road, Andrews, SC 29510 -79.570380  33.462943   
3         127 Schoolhouse Drive, Hemingway, SC 29554 -79.354760  33.678573   
4       

In [6]:
# Load demand data
d_its = {}

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(demand_file, sheet_name=sheet, header=0, index_col=0)
    
    # Fill NaN with 0 if needed
    df = df.fillna(0)
    
    demand_array = df.values

    for i in range(demand_array.shape[0]):
        actual_location = i  # Start location index from 0
        for t in range(demand_array.shape[1]):
            d_its[(actual_location, t, s)] = demand_array[i, t]

# Print a sample to verify the correct location
print("Sample demand data for location 6, time 1, scenario 4:", d_its[(5, 2, 0)])
print("Sample demand data for location 6, time 1, scenario 2:", d_its[(6, 2, 0)])

Sample demand data for location 6, time 1, scenario 4: 286.0581940123904
Sample demand data for location 6, time 1, scenario 2: 0.0


In [7]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
tperiod = 8
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 4
All locations: 14
Warehouse/Supplier locations: 2 [0, 1]
Shelter locations: 4 [2, 3, 4, 5]
RSA locations: 4 [6, 7, 8, 9]
PoD locations: 4 [10, 11, 12, 13]
All resources: 2
Time periods: 6
Time Before Landfall: [0, 1, 2]
Time After Landfall: [3, 4, 5]
Scenario indices: [0]
Scenario probabilities: [1.0]


In [8]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[3, 5]
Number of arcs: 116
Number of arcs eliminated by this arc set: 80


In [9]:
#Defining parameters

W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

#Shelter Capacity at location i when i in V1 q_i[I]
q_i=[]
for i in V1:
    q_i.append(df_V['Pallet Capacity'][i])    

# Initialize inventory list
Inv = []

# Define starting inventory based on instance version
if instance_version == "original":
    for i in range(V):
        Inv.extend([
            df_V['S-Kit StartInv'][i],
            df_V['P-Kit StartInv'][i],
            df_V['Water StartInv'][i],
            df_V['Food StartInv'][i],
            df_V['Cot StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

elif instance_version == "smallsize":
    for i in range(V):
        Inv.append(df_V['Water StartInv'][i])
    I_ir0 = [[inv] for inv in Inv]

elif instance_version == "smallsize2":
    for i in range(V):
        Inv.extend([
            df_V['Water StartInv'][i],
            df_V['Blanket StartInv'][i]
        ])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]


#penalty cost for location i p_i[I]
p_i=[]
for i in range(V):
    p_i.append(df_V['P_i'][i])

#time periods to travel i,j t_ijt[i][j]
t_ijt=[]
for i in range(V):
    row=[]
    for j in range(V):
        t_ijt_hours=timeMatrix[i][j]/3600
        t_ijt_period=math.ceil(t_ijt_hours/tperiod)
        row.append(t_ijt_period)
    t_ijt.append(row)


#distance to travel i,j e_ij[i][j]
e_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        e_ij_miles=distanceMatrix[i][j]*0.000621371
        row.append(e_ij_miles)
    e_ij.append(row)

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r=[]
for r in range(R):
    w_r.append(1/df_R['Pallet Units on Truck'][r])
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter=[]
for r in range(R):
    g_shelter.append(df_R['pallet per S-Kit'][r])

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod=[]
for r in range(R):
    g_pod.append(df_R['pallet per P-Kit'][r])

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i = row['From_Node']
    j = row['To_Node']
    t = row['Time_Period']
    capacity = row['Capacity']
    
    if i not in K_ijt:
        K_ijt[i] = {}
    if j not in K_ijt[i]:
        K_ijt[i][j] = {}
    
    K_ijt[i][j][t] = capacity

# Check the first few entries in K_ijt to ensure it’s structured correctly
#print ("I_ir0 = ", I_ir0)
#print("Inv = ", Inv)
#print("t_ijt = ", t_ijt)
#print("e_ij = ", e_ij)
#print("g_shelter:", g_shelter)
#print("g_pod:", g_pod)
#print("w_r = ", w_r)

# Defining Master Problem

In [10]:
# Create the master problem model
master_model = gp.Model("MasterProblem")

# Define the inventory decision variables I[i, r, t] for first-stage decisions
# These represent the inventory of commodity r at location i at the start of time period t
I = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = master_model.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"I_{i}_{r}_{t}")

# set a very negative initial lower bound            
thetaLB = -1e10

# Define the auxiliary variable θ with an initial lower bound, used for Benders cuts
theta = master_model.addVar(vtype=GRB.CONTINUOUS, lb = thetaLB, name="theta")

for i in range(V):
    for r in range(R):
        master_model.addConstr(I[i, r, 0] == I_ir0[i][r], name=f"initial_inventory_{i}_{r}")
"""
# Add constraints to ensure inventory at warehouses only decreases over time
for i in V0:
    for r in range(R):
        for t in range(1, T):
            master_model.addConstr(
                I[i, r, t] <= I[i, r, t-1],
                name=f"inventory_decrease_warehouse_{i}_{r}_{t}")

# Add constraints to ensure inventory at PoDs only increases before landfall
for i in V3:
    for r in range(R):
        for t in T_Bef[1:]:
            master_model.addConstr(
                I[i, r, t] >= I[i, r, t-1],
                name=f"inventory_increase_pod_{i}_{r}_{t}")
"""
# Set the objective function to minimize θ (representing the expected second-stage cost)
master_model.setObjective(theta, GRB.MINIMIZE)

# Set output flag to 0 to avoid printing Gurobi logs
master_model.setParam("OutputFlag", 0);

# Update the model to integrate new variables
master_model.update()
#master_model.write("master_problem.lp")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-30


# Defining Optimality Subproblem

In [11]:
# Define the optimality subproblem
sub_opt = gp.Model("subprob_optimality")
sub_opt.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the optimality subproblem
U_opt = {}
S_opt = {}
f_opt = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        U_opt[i, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"U_opt_{i}_{t}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        S_opt[i, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"S_opt_{i}_{t}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    f_opt[i, j, r, t] = sub_opt.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"f_opt_{i}_{j}_{r}_{t}")

# Objective: Minimize the expected demand shortage cost
sub_opt.setObjective(gp.quicksum(p_i[i] * U_opt[i, t] for i in range(V) for t in range(T)))

# Constraint for inventory flow balance with U and S
pi_opt_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Start from t=1 since t=0 is the initial condition
        pi_opt_inventory_balance_U[i, t] = sub_opt.addConstr(U_opt[i, t] - U_opt[i, t-1] + S_opt[i, t] == 0, name=f"inventory_balance_U_opt_{i}_{t}")

# Initial shortage for demand at t=0
pi_opt_initial_shortage_U = {}
for i in range(V):
    pi_opt_initial_shortage_U[i] = sub_opt.addConstr(U_opt[i, 0] == 0, name=f"initial_shortage_U_opt_{i}")

# Demand satisfaction constraint
pi_opt_demand_satisfaction = {}
for i in range(V):
    for t in range(T):
        pi_opt_demand_satisfaction[i, t] = sub_opt.addConstr(U_opt[i, t] - S_opt[i, t] >= 0, name=f"demand_satisfaction_opt_{i}_{t}")


# Flow conservation constraint for f with the RHS set to 0 temporarily, using appropriate g_shelter or g_pod
pi_opt_flow_conservation = {}
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            # Add the constraint to the model
            pi_opt_flow_conservation[i, r, t] = sub_opt.addConstr(
                -gp.quicksum(f_opt[i, j, r, t] for j in range(V) if (i, j) in A)
                + gp.quicksum(f_opt[j, i, r, t - t_ijt[j][i] + 1] for j in range(V) if (i, j) in A and t - t_ijt[j][i] + 1 >= 0) #
                - S_opt[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0) == 0,
                name=f"flow_conservation_opt_{i}_{r}_{t}")

# Inventory balance constraint with RHS set to 0 for now
pi_opt_inventory_balance = {}
for i in range(V):
    for r in range(R):
        for t in range(T):       
            # Add the constraint to the model
            pi_opt_inventory_balance[i, r, t] = sub_opt.addConstr(
                -gp.quicksum(f_opt[i, j, r, t] for j in range(V) if (i, j) in A) #
                - S_opt[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0) >= 0,
                name=f"inventory_balance_opt_{i}_{r}_{t}")


# Arc capacity constraint
pi_opt_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                # Check if the capacity for this (i, j, t) exists in K_ijt
                #if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                    pi_opt_arc_capacity[i, j, t] = sub_opt.addConstr(-gp.quicksum(w_r[r] * f_opt[i, j, r, t] for r in range(R)) >= -K_ijt[i][j][t],
                        name=f"arc_capacity_opt_{i}_{j}_{t}")

sub_opt.setParam("OutputFlag", 0);

# Defining Feasibility Subproblem

In [12]:
# Define the feasibility subproblem
sub_feas = gp.Model("subprob_feasibility")
sub_feas.modelSense = GRB.MINIMIZE

# Define second-stage decision variables U, S, and f for the feasibility subproblem
U_feas = {}
S_feas = {}
f_feas = {}

# U: Demand shortage at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        U_feas[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"U_feas_{i}_{t}")

# S: Demand satisfied at location i at time period t in scenario s
for i in range(V):
    for t in range(T):
        S_feas[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"S_feas_{i}_{t}")

# f: Quantity of commodity r traveling on link (i, j) at time period t in scenario s
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for r in range(R):
                for t in range(T):
                    f_feas[i, j, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, obj=0, name=f"f_feas_{i}_{j}_{r}_{t}")


# Now define slack variables ε (epsilon) for each group in the objective function
# ε variables with objective coefficients corresponding to their respective constraint group
epsilon_1 = {}
for i in range(V):
    for t in range(1, T):
        epsilon_1[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon1_{i}_{t}")

epsilon_2 = {}
for i in range(V):
    for t in range(1, T):
        epsilon_2[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon2_{i}_{t}")

epsilon_3 = {}
for i in range(V):
    for t in range(T):
        epsilon_3[i, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon3_{i}_{t}")

epsilon_4 = {}
for i in range(V):
    for r in range(R):
        for t in range(T-1):
            epsilon_4[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon4_{i}_{r}_{t}")

epsilon_5 = {}
for i in range(V):
    for r in range(R):
        for t in range(T-1):
            epsilon_5[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon5_{i}_{r}_{t}")

epsilon_6 = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            epsilon_6[i, r, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon6_{i}_{r}_{t}")

epsilon_7 = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:
            for t in range(T):
                epsilon_7[i, j, t] = sub_feas.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"epsilon7_{i}_{j}_{t}")

sub_feas.setObjective(
    gp.quicksum(epsilon_1[i, t] + epsilon_2[i, t] for i in range(V) for t in range(1, T)) +
    gp.quicksum(epsilon_3[i, t] for i in range(V) for t in range(T)) +
    gp.quicksum(epsilon_4[i, r, t] + epsilon_5[i, r, t] for i in range(V) for r in range(R) for t in range(T - 1)) +
    gp.quicksum(epsilon_6[i, r, t] for i in range(V) for r in range(R) for t in range(T)) +
    gp.quicksum(epsilon_7[i, j, t] for i in range(V) for j in range(V) if (i, j) in A for t in range(T)))

# Constraint: Inventory flow balance for U, S, and epsilon variables
pi_feas_inventory_balance_U = {}
for i in range(V):
    for t in range(1, T):  # Note: starting from t=1 as per the constraint
        pi_feas_inventory_balance_U[i, t] = sub_feas.addConstr(
            U_feas[i, t] - U_feas[i, t-1] + S_feas[i, t] + epsilon_1[i, t] - epsilon_2[i, t] == 0,
            name=f"inventory_balance_U_feas_{i}_{t}"
        )

# Initialize shortage for U at t=0
pi_feas_initial_shortage_U = {}
for i in range(V):
    pi_feas_initial_shortage_U[i] = sub_feas.addConstr(
        U_feas[i, 0] == 0,
        name=f"initial_shortage_U_feas_{i}"
    )


# Demand satisfaction constraint
pi_feas_demand_satisfaction = {}
for i in range(V):
    for t in range(T):
        pi_feas_demand_satisfaction[i, t] = sub_feas.addConstr(
            U_feas[i, t] - S_feas[i, t] + epsilon_3[i, t] >= 0,
            name=f"demand_satisfaction_feas_{i}_{t}"
        )

# Flow conservation constraint
pi_feas_flow_conservation = {}
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            pi_feas_flow_conservation[i, r, t] = sub_feas.addConstr(
                -gp.quicksum(f_feas[i, j, r, t] for j in range(V) if (i, j) in A)
                + gp.quicksum(f_feas[j, i, r, t - t_ijt[j][i] + 1] for j in range(V) if (i, j) in A and t - t_ijt[j][i] + 1 >= 0) #
                - S_feas[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0)
                + epsilon_4[i, r, t] - epsilon_5[i, r, t] == 0,
                name=f"flow_conservation_feas_{i}_{r}_{t}"
            )

# Constraint for flow and demand satisfaction with epsilon_6
pi_feas_inventory_balance = {}
for i in range(V):
    for r in range(R):
        for t in range(T):
            pi_feas_inventory_balance[i, r, t] = sub_feas.addConstr(
                -gp.quicksum(f_feas[i, j, r, t] for j in range(V) if (i, j) in A)
                - S_feas[i, t] * (g_shelter[r] if i in V1 else g_pod[r] if i in V3 else 0)
                + epsilon_6[i, r, t] >= 0,  # Right-hand side set to 0 for now
                name=f"inventory_balance_feas_{i}_{r}_{t}"
            )

# Flow capacity constraint for all links, time periods, and scenarios
pi_feas_arc_capacity = {}
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                # Check if K_ijt[i][j][t] exists before adding the constraint
                #if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                    pi_feas_arc_capacity[i, j, t] = sub_feas.addConstr(
                        -gp.quicksum(w_r[r] * f_feas[i, j, r, t] for r in range(R))
                        + epsilon_7[i, j, t] >= -K_ijt[i][j][t],
                        name=f"arc_capacity_feas_{i}_{j}_{t}"
                    )

sub_feas.setParam("OutputFlag", 0);

# Creating Benders Loop

In [13]:
LB = thetaLB
UB = 1e10

Ocuts = 0
Fcuts = 0

iter = 0
eps = 1e-5  # tolerance parameter

while (UB - LB) / max(1e-10,abs(UB)) > eps:
    iter += 1
    #if iter > 5:
     #   break
    
    # First, solve the master problem to get the values of first-stage variables and theta
    I_values = {}  # Store I values for each (i, r, t)
    theta_value = 0  # Initialize theta value

    # Optimize the master problem
    master_model.optimize()

    # Check if the optimization was successful
    if master_model.status == GRB.OPTIMAL:
        # Extract the values of first-stage decision variables I[i, r, t]
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    I_values[i, r, t] = I[i, r, t].x

        # Get the value of theta
        theta_value = theta.x
        """
        # Extract the value of theta
        print(f"Iteration {iter}:")
        print(f"  Objective value (Lower Bound): {master_model.objVal}")
        print(f"  Theta value: {theta.x}")
    
        # Print the values of first-stage decision variables I[i, r, t]
        print("  First-stage decision variables (I[i, r, t]):")
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    print(f"    I[{i}, {r}, {t}] = {I[i, r, t].x}")
        """
        # Update the lower bound
        LB = master_model.objVal       
         
    elif master_model.status == GRB.INFEASIBLE:
        print("Master problem is infeasible.")
        exit(0)  # Or handle infeasibility as required for your problem

    elif master_model.status == GRB.UNBOUNDED:
        print("Master problem is unbounded.")
        exit(0)  # Or handle unboundedness as required for your problem

    else:
        print(f"Optimization was stopped with status {master_model.status}")
        exit(0)
    
    # Number of scenarios
    num = len(sce)

    # Optimality subproblem dual multipliers
    pi_opt_inventory_balance_U_val = {s: {} for s in range(num)}
    pi_opt_initial_shortage_U_val = {s: {} for s in range(num)}
    pi_opt_demand_satisfaction_val = {s: {} for s in range(num)}
    pi_opt_flow_conservation_val = {s: {} for s in range(num)}
    pi_opt_inventory_balance_val = {s: {} for s in range(num)}
    pi_opt_arc_capacity_val = {s: {} for s in range(num)}
    
    # Feasibility subproblem dual multipliers
    pi_feas_inventory_balance_U_val = {}
    pi_feas_initial_shortage_U_val = {}
    pi_feas_demand_satisfaction_val = {}
    pi_feas_flow_conservation_val = {}
    pi_feas_inventory_balance_val = {}
    pi_feas_arc_capacity_val = {}

    # Initialize a vector to store the optimal objective value for the optimality subproblem for each scenario
    Q_opt = np.zeros(num)
    # Initialize a vector to store the optimal objective value for the feasibility subproblem for each scenario
    w_feas = np.zeros(num)
    FLAG = 0; #(0 if I_hat is feasible ∀ s, and 1 otherwise)

    # Update RHS for optimality subproblem constraints for every scenario
    infeasible_sce = -1
    for s in range(num):  # Loop over all scenarios
        # Update RHS of inventory flow balance constraints
        for i in range(V):
            for t in range(1, T):
                pi_opt_inventory_balance_U[i, t].setAttr(GRB.Attr.RHS, d_its[(i, t, s)])
        
        # Update RHS of initial shortage constraints
        for i in range(V):
            pi_opt_initial_shortage_U[i].setAttr(GRB.Attr.RHS, d_its[(i, 0, s)])
        
        # Update RHS of flow conservation constraints
        for i in range(V):
            for r in range(R):
                for t in range(T - 1):
                    pi_opt_flow_conservation[i, r, t].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])
        
        # Update RHS of inventory balance constraints
        for i in range(V):
            for r in range(R):
                for t in range(T):
                    pi_opt_inventory_balance[i, r, t].setAttr(GRB.Attr.RHS, -I_values[i, r, t])
        

        sub_opt.optimize()    
        if sub_opt.status == GRB.OPTIMAL:
            # Update the optimal objective value for each scenario
            Q_opt[s] = sub_opt.objVal
    
            # Retrieve and store dual values for each relevant constraint
            for key, constraint in pi_opt_inventory_balance_U.items():
                pi_opt_inventory_balance_U_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_initial_shortage_U.items():
                pi_opt_initial_shortage_U_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_demand_satisfaction.items():
                pi_opt_demand_satisfaction_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_flow_conservation.items():
                pi_opt_flow_conservation_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_inventory_balance.items():
                pi_opt_inventory_balance_val[s][key] = constraint.pi
        
            for key, constraint in pi_opt_arc_capacity.items():
                pi_opt_arc_capacity_val[s][key] = constraint.pi
        
        elif sub_opt.status == GRB.INFEASIBLE:
            # Set FLAG to 1 if infeasible, indicating the need for a feasibility cut
            FLAG = 1
            infeasible_sce = s
            break;
        else:
            # Handle other statuses
            print('One of the scenario subproblems is infeasible')
            print('status =', sub_opt.status)
            exit(0)
    print("FLAG =", FLAG)
    if FLAG == 1:
        
            # Update RHS for feasibility subproblem constraints for every scenario
            s = infeasible_sce
            
            # Update RHS of inventory flow balance constraints for U
            for i in range(V):
                for t in range(1, T):
                    pi_feas_inventory_balance_U[i, t].setAttr(GRB.Attr.RHS, d_its[(i, t, s)])
            
            # Update RHS of initial shortage constraints for U at t=0
            for i in range(V):
                pi_feas_initial_shortage_U[i].setAttr(GRB.Attr.RHS, d_its[(i, 0, s)])
            
            
            # Update RHS of flow conservation constraints
            for i in range(V):
                for r in range(R):
                    for t in range(T - 1):
                        pi_feas_flow_conservation[i, r, t].setAttr(GRB.Attr.RHS, I_values[i, r, t+1] - I_values[i, r, t])
            
            # Update RHS of inventory balance constraints
            for i in range(V):
                for r in range(R):
                    for t in range(T):
                        pi_feas_inventory_balance[i, r, t].setAttr(GRB.Attr.RHS, -I_values[i, r, t])
                                

            # Optimize the feasibility subproblem
            sub_feas.optimize()
            w_feas[s] = sub_feas.objVal  # Store the optimal objective value for this scenario
            print("sub_feas.objval = ", sub_feas.objVal)

            # Retrieve and store dual values for each relevant constraint in the feasibility subproblem
            for key, constraint in pi_feas_inventory_balance_U.items():
                pi_feas_inventory_balance_U_val[key] = constraint.pi
            for key, constraint in pi_feas_initial_shortage_U.items():
                pi_feas_initial_shortage_U_val[key] = constraint.pi
            for key, constraint in pi_feas_demand_satisfaction.items():
                pi_feas_demand_satisfaction_val[key] = constraint.pi            
            for key, constraint in pi_feas_flow_conservation.items():
                pi_feas_flow_conservation_val[key] = constraint.pi            
            for key, constraint in pi_feas_inventory_balance.items():
                pi_feas_inventory_balance_val[key] = constraint.pi            
            for key, constraint in pi_feas_arc_capacity.items():
                pi_feas_arc_capacity_val[key] = constraint.pi   
            
            if w_feas[s] > eps:
                # Initialize the feasibility cut expression
                feasibility_cut_expr = gp.LinExpr()
            
                # Add the terms for inventory flow balance for U and S constraints
                for key, dual_val in pi_feas_inventory_balance_U_val.items():
                    i, t = key
                    feasibility_cut_expr += dual_val * d_its[(i, t, s)]
                    
                # Add the terms for initial shortage for U constraints
                for key, dual_val in pi_feas_initial_shortage_U_val.items():
                    i = key
                    feasibility_cut_expr += dual_val * d_its[(i, 0, s)]
                
                # Add the terms for demand satisfaction constraints
                for key, dual_val in pi_feas_demand_satisfaction_val.items():
                    i, t = key
                    feasibility_cut_expr += dual_val * 0

                # Add the terms for flow conservation constraints
                for key, dual_val in pi_feas_flow_conservation_val.items():
                    i, r, t = key
                    feasibility_cut_expr += dual_val * (I[i, r, t+1] - I[i, r, t])
    
                # Add the terms for inventory balance constraints
                for key, dual_val in pi_feas_inventory_balance_val.items():
                    i, r, t = key
                    feasibility_cut_expr += dual_val * (-I[i, r, t])

                # Add the terms for arc capacity constraints
                for key, dual_val in pi_feas_arc_capacity_val.items():
                    i, j, t = key
                    feasibility_cut_expr += dual_val * -K_ijt[i][j][t]
                """
                #Evaluate the LHS of the feasibility cut
                lhs_value = feasibility_cut_expr.getValue()
                
                #Compare LHS with the feasibility problem's objective value
                print(f"Iteration {iter}, Scenario {s}")
                print(f"Feasibility Problem Objective Value: {w_feas[s]}")
                print(f"LHS of Feasibility Cut: {lhs_value}")
                print(f"Difference: {abs(lhs_value - w_feas[s])}")
                """
                #print("feasibility cut:", feasibility_cut_expr)
                # Add the feasibility cut to the master problem
                master_model.addConstr(feasibility_cut_expr <= 0, name=f"feasibility_cut_scenario_{s}")
                Fcuts += 1  # Increment the counter for feasibility cuts
                #master_model.write(f"master_problem_with_cuts_iter_{iter}.lp")
                

    else:
        print("here")
        print("Q_opt = ", Q_opt)
        print("w_feas =", w_feas)
        # Calculate the temporary upper bound using only the second-stage costs
        tempUB = sum(Q_opt[s] * P[s] for s in range(num))        
        if tempUB < UB:
            UB = tempUB
        if (UB - LB) / max(1e-10, abs(LB)) < eps:
            break

        else:
            # Initialize the optimality cut expression
            optimality_cut_expr = gp.LinExpr()
            for s in range(num):
                 
                # Add the terms for inventory flow balance with U and S constraints
                for key, dual_val in pi_opt_inventory_balance_U_val[s].items():
                    i, t = key
                    optimality_cut_expr += P[s] * dual_val * d_its[(i, t, s)]
                
                # Add the terms for initial shortage for U constraints
                for key, dual_val in pi_opt_initial_shortage_U_val[s].items():
                    i = key
                    optimality_cut_expr += P[s] * dual_val * d_its[(i, 0, s)]
                
                # Add the terms for demand satisfaction constraints
                for key, dual_val in pi_opt_demand_satisfaction_val[s].items():
                    i, t = key
                    optimality_cut_expr += P[s] * dual_val * 0
                
                # Add the terms for flow conservation constraints
                for key, dual_val in pi_opt_flow_conservation_val[s].items():
                    i, r, t = key
                    optimality_cut_expr += P[s] * dual_val * (I[i, r, t + 1] - I[i, r, t])
                
                # Add the terms for inventory balance constraints
                for key, dual_val in pi_opt_inventory_balance_val[s].items():
                    i, r, t = key
                    optimality_cut_expr += P[s] * dual_val * (-I[i, r, t])
                
                # Add the terms for arc capacity constraints
                for key, dual_val in pi_opt_arc_capacity_val[s].items():
                    i, j, t = key
                    optimality_cut_expr += P[s] * dual_val * -K_ijt[i][j][t]

                #print("optimality cut expression:", optimality_cut_expr)
            #print("optimality cut:", optimality_cut_expr)
            
            # Add the optimality cut to the master model
            master_model.addConstr(theta >= optimality_cut_expr, name="optimality_cut")
            Ocuts += 1  # Increment the counter for optimality cuts

            

    # Print information at each iteration
    print("iter =", iter)
    print("LB =", LB)
    print("UB =", UB)
    print("===============================================================")


# Final summary after convergence
print("#########################################################################################")
print("#########################################################################################")
print("Final iteration =", iter)
print("Number of feasibility cuts added =", Fcuts)
print("Number of optimality cuts added =", Ocuts)
print("Lower Bound (LB) =", LB)
print("Upper Bound (UB) =", UB)
print("FLAG =", FLAG)


# Print first-stage decision variable values (I_values)
#print("I_values (First-stage decision variables):")
#for key, value in I_values.items():
    #print(f"I[{key}] =", value)

FLAG = 1
sub_feas.objval =  590.0
iter = 1
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  590.0
iter = 2
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  885.0
iter = 3
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  990.0
iter = 4
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1250.0
iter = 5
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  902.5
iter = 6
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1327.5
iter = 7
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  980.0
iter = 8
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1966.6666666666665
iter = 9
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1180.0
iter = 10
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1770.0
iter = 11
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1877.2727272727275
iter = 12
LB = -10000000000.0
UB

FLAG = 1
sub_feas.objval =  2241.016286823082
iter = 65
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  2851.754631278764
iter = 66
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  529.047619047619
iter = 67
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  780.0
iter = 68
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1152.3529411764705
iter = 69
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  2560.0
iter = 70
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  4722.7692307692305
iter = 71
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1505.26485466861
iter = 72
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1243.7706659012629
iter = 73
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  2051.1218119352534
iter = 74
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  2094.685834747468
iter = 75
LB = -100000000

sub_feas.objval =  461.66666666666674
iter = 118
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  992.0
iter = 119
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1205.3559322033898
iter = 120
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  400.0
iter = 121
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1629.64705882353
iter = 122
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  409.44444444444446
iter = 123
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  666.6666666666665
iter = 124
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  494.5860583724286
iter = 125
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  854.455355974885
iter = 126
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1055.9112606187082
iter = 127
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  468.9552841483637
iter = 128
LB = -10000000

FLAG = 1
sub_feas.objval =  507.99687688009914
iter = 170
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  352.5301711303697
iter = 171
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  454.3299374338597
iter = 172
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  589.4719297983117
iter = 173
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  954.1785985749945
iter = 174
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  724.702828378158
iter = 175
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  820.0
iter = 176
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  372.22287871638144
iter = 177
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  577.2228787163813
iter = 178
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  1231.8072770174222
iter = 179
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  536.5269461077843
iter

FLAG = 1
sub_feas.objval =  505.9423133440671
iter = 241
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  793.6988518837105
iter = 242
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  337.78517091700587
iter = 243
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  390.4018998462436
iter = 244
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  458.4295278025798
iter = 245
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  403.8695377091673
iter = 246
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  398.8537091649881
iter = 247
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  492.69790144087386
iter = 248
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  296.48375689737463
iter = 249
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  403.51193484120085
iter = 250
LB = -10000000000.0
UB = 10000000000.0
FLAG = 1
sub_feas.objval =  17.52931

FLAG = 1
sub_feas.objval =  3100918450.79897
iter = 311
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  590085267.1233664
iter = 312
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  1439363197.7112799
iter = 313
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  577534266.5658276
iter = 314
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  351558612.6156108
iter = 315
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  248903408.83365196
iter = 316
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  236458765.12117726
iter = 317
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  224549413.4669246
iter = 318
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  1185727453.5444264
iter = 319
LB = -10000000000.0
UB = 7902.744270809975
FLAG = 1
sub_feas.objval =  618708844.0344137
iter = 320
LB = -10000000000.0
UB = 7902.7442708099

... (72 outputs left out)

FLAG = 1
sub_feas.objval =  1.0959906111841256
iter = 4182
LB = 3983.7821251416285
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  1.0074464274776764
iter = 4183
LB = 3983.7821251416035
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.9582965814351059
iter = 4184
LB = 3983.7821251416153
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.9582965814350822
iter = 4185
LB = 3983.7821251415917
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.9633150631066485
iter = 4186
LB = 3983.7821251416085
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  1.5151861454507065
iter = 4187
LB = 3983.782125141599
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  1.5151861454502902
iter = 4188
LB = 3983.7821251416053
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  1.7070568443536347
iter = 4189
LB = 3983.782125141598
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  1.5663266500167419
iter = 4190
LB = 3983.782125141614
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  2.4819658225

FLAG = 1
sub_feas.objval =  3.8505411626360435
iter = 4231
LB = 4025.3369073801264
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  3.9478578184309203
iter = 4232
LB = 4025.3369073801305
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  3.796017133106755
iter = 4233
LB = 4025.336907380149
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  3.3548086210502244
iter = 4234
LB = 4025.336907380128
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  3.1104301705855093
iter = 4235
LB = 4025.336907380137
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  3.1104301705842228
iter = 4236
LB = 4025.336907380161
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  2.305746954750747
iter = 4237
LB = 4025.3369073801387
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  2.2996769452388643
iter = 4238
LB = 4025.336907380136
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  2.241542780864513
iter = 4239
LB = 4025.336907380132
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  2.2207824363828674

FLAG = 1
sub_feas.objval =  0.19089817618281194
iter = 4282
LB = 4025.7416678327168
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.22262751604998954
iter = 4283
LB = 4025.7416678327404
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.17784594737458048
iter = 4284
LB = 4025.741667832705
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.17784594737458556
iter = 4285
LB = 4025.7416678327027
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.23387135756201727
iter = 4286
LB = 4025.7416678327295
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.17682061474503985
iter = 4287
LB = 4025.7416678326904
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.18339135456463918
iter = 4288
LB = 4025.74166783269
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.20061717036465498
iter = 4289
LB = 4025.7416678327118
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.18471528905791224
iter = 4290
LB = 4025.741667832715
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.18

FLAG = 1
sub_feas.objval =  0.5376137428045011
iter = 4331
LB = 4031.209185307556
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.5261925100301054
iter = 4332
LB = 4031.2091853075663
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.5413715288447192
iter = 4333
LB = 4031.209185307575
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.5261925100302305
iter = 4334
LB = 4031.2091853074667
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.5261925100277459
iter = 4335
LB = 4031.209185307437
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.5307244214975694
iter = 4336
LB = 4033.8780996463297
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.5126378866113394
iter = 4337
LB = 4033.878099646279
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.4995718774742839
iter = 4338
LB = 4033.878099646283
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.4865342152838114
iter = 4339
LB = 4033.8780996462565
UB = 4034.5523870210923
FLAG = 1
sub_feas.objval =  0.172165764936